In [25]:
import os
import glob
import time
import warnings
import numpy as np
import pandas as pd
import joblib
import re

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

warnings.filterwarnings('ignore')


In [26]:
def build_preprocessor(df: pd.DataFrame) -> ColumnTransformer:
    bin_cols = [c for c in df.columns if re.match(r'obj\d+_bin_\d+', c)]
    nom_cols = [c for c in df.columns if re.match(r'obj\d+_nom_\d+', c)]
    ord_cols = [c for c in df.columns if re.match(r'obj\d+_ord_\d+', c)]
    qnt_cols = [c for c in df.columns if re.match(r'obj\d+_qnt_\d+', c)]
    
    transformers = []
    if bin_cols: 
        transformers.append(('bin', 'passthrough', bin_cols))
    if qnt_cols: 
        transformers.append(('qnt', StandardScaler(), qnt_cols))
    if nom_cols: 
        transformers.append(('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nom_cols))
    if ord_cols: 
        transformers.append(('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ord_cols))
    
    return ColumnTransformer(transformers, remainder='drop')


In [27]:
dataset_files = sorted(glob.glob('datasets/*.csv'))

def get_n_samples(fname):
    match = re.search(r'samples(\d+)', os.path.basename(fname))
    return int(match.group(1)) if match else 0

dataset_files.sort(key=get_n_samples)

smallest_file = dataset_files[0]
largest_file = dataset_files[-1]


Задание 1

In [28]:
import pygad

In [ ]:
        
MODELS_GA = {
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    'DecisionTree': DecisionTreeClassifier(random_state=42, class_weight='balanced')
}

HYPERPARAMS_SPACE_GA = {
    'LogisticRegression': {
        'gene_space': [{'low': -2, 'high': 2}], 
        'apply_params': lambda params: {'C': 10 ** params[0]},
        'gene_type': float
    },
    'DecisionTree': {
        'gene_space': [
            {'low': 2, 'high': 50}, 
            {'low': 2, 'high': 20}
        ],
        'apply_params': lambda params: {
            'max_depth': int(params[0]) if int(params[0]) < 50 else None, 
            'min_samples_split': int(params[1])
        },
        'gene_type': int
    }
}


In [30]:
def create_ga_fitness(model_template, X_train, y_train, X_val, y_val, model_name):
    space = HYPERPARAMS_SPACE_GA[model_name]
    
    def fitness_func(ga_instance, solution, solution_idx):
        try:
            params = space['apply_params'](solution)
            model_inst = clone(model_template)
            model_inst.set_params(**params)
            
            preprocessor = build_preprocessor(X_train)
            pipe = Pipeline([('pre', preprocessor), ('clf', model_inst)])
            
            pipe.fit(X_train, y_train)
            y_pred = pipe.predict(X_val)
            score = f1_score(y_val, y_pred, pos_label='Да', zero_division=0)
            return score
        except Exception as e:
            return 0.0
    
    return fitness_func


In [31]:
results_ga = []
best_models_ga = {}

for ds_path in [smallest_file, largest_file]:
    df = pd.read_csv(ds_path)
    X = df.drop('Collision', axis=1)
    y = df['Collision']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_subtrain, X_val, y_subtrain, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)
    
    for name, model_tpl in MODELS_GA.items():
        
        fitness_func = create_ga_fitness(model_tpl, X_subtrain, y_subtrain, X_val, y_val, name)
        num_genes = len(HYPERPARAMS_SPACE_GA[name]['gene_space'])
        
        ga_instance = pygad.GA(
            num_generations=10,
            num_parents_mating=4,
            sol_per_pop=10,
            num_genes=num_genes,
            gene_space=HYPERPARAMS_SPACE_GA[name]['gene_space'],
            fitness_func=fitness_func,
            stop_criteria="reach_0.95",
            random_seed=42,
            suppress_warnings=True
        )
        
        start_time = time.perf_counter()
        ga_instance.run()
        end_time = time.perf_counter()
        
        best_solution, best_fitness, _ = ga_instance.best_solution()
        best_params = HYPERPARAMS_SPACE_GA[name]['apply_params'](best_solution)
        
        final_model = clone(model_tpl)
        final_model.set_params(**best_params)
        pipe = Pipeline([('pre', build_preprocessor(X_train)), ('clf', final_model)])
        pipe.fit(X_train, y_train)
        
        y_pred = pipe.predict(X_test)
        f1_test = f1_score(y_test, y_pred, pos_label='Да', zero_division=0)
        
        elapsed = end_time - start_time
        
        print(f"    Параметры: {best_params}")
        print(f"    Время поиска: {elapsed:.2f} сек")
        print(f"    Test F1: {f1_test:.4f}")
        
        results_ga.append({
            'Lab': 'GA',
            'Dataset': os.path.basename(ds_path),
            'Model': name,
            'Search_Time_sec': round(elapsed, 4),
            'Test_F1': round(f1_test, 4)
        })
        
        if ds_path == largest_file:
            best_models_ga[name] = pipe

os.makedirs('models_ga', exist_ok=True)
for name, model in best_models_ga.items():
    path = f'models_ga/{name}_ga_tuned.joblib'
    joblib.dump(model, path)

df_results_ga = pd.DataFrame(results_ga)
print("\nРезультаты GA:")
print(df_results_ga.to_string(index=False))

    Параметры: {'C': np.float64(35.387588647792384)}
    Время поиска: 2.09 сек
    Test F1: 0.5000
    Параметры: {'max_depth': 41, 'min_samples_split': 6}
    Время поиска: 1.10 сек
    Test F1: 0.9231
    Параметры: {'C': np.float64(1.2030178871154662)}
    Время поиска: 2.61 сек
    Test F1: 0.6964
    Параметры: {'max_depth': 40, 'min_samples_split': 3}
    Время поиска: 2.29 сек
    Test F1: 0.8571

Результаты GA:
Lab                           Dataset              Model  Search_Time_sec  Test_F1
 GA    dataset_01_samples50_feat6.csv LogisticRegression           2.0934   0.5000
 GA    dataset_01_samples50_feat6.csv       DecisionTree           1.1008   0.9231
 GA dataset_12_samples1200_feat12.csv LogisticRegression           2.6141   0.6964
 GA dataset_12_samples1200_feat12.csv       DecisionTree           2.2919   0.8571


Задание 2

In [32]:
import pyswarms as ps

In [33]:
MODELS_PSO = MODELS_GA

BOUNDS_PSO = {
    'LogisticRegression': {
        'bounds': (np.array([-2.0]), np.array([2.0])),
        'decode': lambda x: {'C': 10**x[0]}
    },
    'DecisionTree': {
        'bounds': (np.array([2.0, 2.0]), np.array([50.0, 20.0])),
        'decode': lambda x: {
            'max_depth': int(np.round(x[0])) if int(np.round(x[0])) < 50 else None,
            'min_samples_split': int(np.round(x[1]))
        }
    }
}

OPTIONS_PSO = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}


In [34]:
def create_pso_objective(model_template, X_train, y_train, X_val, y_val, model_name):
    decode_func = BOUNDS_PSO[model_name]['decode']
    
    def objective(X_particles):
        n_particles = X_particles.shape[0]
        scores = np.zeros(n_particles)
        
        for i in range(n_particles):
            try:
                params = decode_func(X_particles[i])
                model_inst = clone(model_template)
                model_inst.set_params(**params)
                
                preprocessor = build_preprocessor(X_train)
                pipe = Pipeline([('pre', preprocessor), ('clf', model_inst)])
                
                pipe.fit(X_train, y_train)
                y_pred = pipe.predict(X_val)
                score = f1_score(y_val, y_pred, pos_label='Да', zero_division=0)
                scores[i] = -score
            except:
                scores[i] = 0.0
        return scores
    
    return objective


In [35]:
results_pso = []
best_models_pso = {}

for ds_path in [smallest_file, largest_file]:
    df = pd.read_csv(ds_path)
    X = df.drop('Collision', axis=1)
    y = df['Collision']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_subtrain, X_val, y_subtrain, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)
    
    for name, model_tpl in MODELS_PSO.items():
        
        bounds = BOUNDS_PSO[name]['bounds']
        obj_func = create_pso_objective(model_tpl, X_subtrain, y_subtrain, X_val, y_val, name)
        
        n_dim = len(bounds[0])
        swarm = ps.single.GlobalBestPSO(
            n_particles=10,
            dimensions=n_dim,
            options=OPTIONS_PSO,
            bounds=bounds
        )
        
        start_time = time.perf_counter()
        cost, pos = swarm.optimize(obj_func, iters=10)
        end_time = time.perf_counter()
        
        elapsed = end_time - start_time
        best_params = BOUNDS_PSO[name]['decode'](pos)
        
        final_model = clone(model_tpl)
        final_model.set_params(**best_params)
        pipe = Pipeline([('pre', build_preprocessor(X_train)), ('clf', final_model)])
        pipe.fit(X_train, y_train)
        
        y_pred = pipe.predict(X_test)
        f1_test = f1_score(y_test, y_pred, pos_label='Да', zero_division=0)
        
        print(f"    Параметры: {best_params}")
        print(f"    Время поиска: {elapsed:.2f} сек")
        print(f"    Test F1: {f1_test:.4f}")
        
        results_pso.append({
            'Lab': 'PSO',
            'Dataset': os.path.basename(ds_path),
            'Model': name,
            'Search_Time_sec': round(elapsed, 4),
            'Test_F1': round(f1_test, 4)
        })
        
        if ds_path == largest_file:
            best_models_pso[name] = pipe

os.makedirs('models_pso', exist_ok=True)
for name, model in best_models_pso.items():
    path = f'models_pso/{name}_pso_tuned.joblib'
    joblib.dump(model, path)

df_results_pso = pd.DataFrame(results_pso)
print("\nРезультаты PSO:")
print(df_results_pso.to_string(index=False))

2026-04-11 16:27:13,500 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|10/10, best_cost=-0.8
2026-04-11 16:27:15,591 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.8, best pos: [1.41115304]
2026-04-11 16:27:15,625 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}


    Параметры: {'C': np.float64(25.7722916520659)}
    Время поиска: 2.09 сек
    Test F1: 0.5000


pyswarms.single.global_best: 100%|██████████|10/10, best_cost=-0.769
2026-04-11 16:27:16,888 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.7692307692307693, best pos: [35.64651831  3.30973411]
2026-04-11 16:27:16,915 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}


    Параметры: {'max_depth': 36, 'min_samples_split': 3}
    Время поиска: 1.26 сек
    Test F1: 0.9231


pyswarms.single.global_best: 100%|██████████|10/10, best_cost=-0.756
2026-04-11 16:27:19,673 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.7564469914040115, best pos: [-0.06085357]
2026-04-11 16:27:19,720 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}


    Параметры: {'C': np.float64(0.8692534545411952)}
    Время поиска: 2.77 сек
    Test F1: 0.6964


pyswarms.single.global_best: 100%|██████████|10/10, best_cost=-0.857
2026-04-11 16:27:21,895 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.8571428571428571, best pos: [30.3341394   3.19680465]


    Параметры: {'max_depth': 30, 'min_samples_split': 3}
    Время поиска: 2.18 сек
    Test F1: 0.8571

Результаты PSO:
Lab                           Dataset              Model  Search_Time_sec  Test_F1
PSO    dataset_01_samples50_feat6.csv LogisticRegression           2.0921   0.5000
PSO    dataset_01_samples50_feat6.csv       DecisionTree           1.2646   0.9231
PSO dataset_12_samples1200_feat12.csv LogisticRegression           2.7660   0.6964
PSO dataset_12_samples1200_feat12.csv       DecisionTree           2.1770   0.8571


In [ ]:

df_comparison = pd.concat([df_results_ga, df_results_pso], ignore_index=True)

pivot_table = df_comparison.pivot_table(
    index=['Dataset', 'Model'],
    columns='Lab',
    values=['Search_Time_sec', 'Test_F1'],
    aggfunc='first'
).round(4)

print("\nСравнение GA и PSO:")
print(pivot_table)

avg_times = df_comparison.groupby('Lab')['Search_Time_sec'].mean()
avg_f1s = df_comparison.groupby('Lab')['Test_F1'].mean()







Сравнение GA и PSO:
                                                     Search_Time_sec          \
Lab                                                               GA     PSO   
Dataset                           Model                                        
dataset_01_samples50_feat6.csv    DecisionTree                1.1008  1.2646   
                                  LogisticRegression          2.0934  2.0921   
dataset_12_samples1200_feat12.csv DecisionTree                2.2919  2.1770   
                                  LogisticRegression          2.6141  2.7660   

                                                     Test_F1          
Lab                                                       GA     PSO  
Dataset                           Model                               
dataset_01_samples50_feat6.csv    DecisionTree        0.9231  0.9231  
                                  LogisticRegression  0.5000  0.5000  
dataset_12_samples1200_feat12.csv DecisionTree        0.8571  0

In [38]:
print("\nСреднее время поиска:")
avg_times


Среднее время поиска:


Lab
GA     2.025050
PSO    2.074925
Name: Search_Time_sec, dtype: float64

In [39]:
print("\nСредний F1-score:")
avg_f1s


Средний F1-score:


Lab
GA     0.74415
PSO    0.74415
Name: Test_F1, dtype: float64